# Bounding Box selber festlegen mit automatischem Export als GeoJSON und Variable (openEO-Format)

In [ ]:
%pip install folium geopandas leafmap matplotlib openeo pyproj rasterio shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.2/669.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.2/349.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.

In [ ]:
import folium
import geopandas as gpd
import json
import leafmap
import math
import matplotlib.pyplot as plt
import os
import openeo
import pyproj
import rasterio

from rasterio.plot import show
from shapely.geometry import Polygon

In [ ]:
# NUR IN GOOGLE COLAB RELEVANT; DESWEGEN AUSKOMMENTIERT
# from google.colab import output
# output.enable_custom_widget_manager()

In [ ]:
# NUR IN GOOGLE COLAB RELEVANT; DESWEGEN AUSKOMMENTIERT
# from google.colab import output
# output.disable_custom_widget_manager()

In [ ]:
m = leafmap.Map(center=[50.0, 10.0], zoom=4)
m

Map(center=[50.0, 10.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [ ]:
if m.user_roi:
    # Koordinaten extrahieren
    coords = m.user_roi['geometry']['coordinates'][0]
    lons, lats = zip(*coords)

    # Bounding Box berechnen
    west, south, east, north = min(lons), min(lats), max(lons), max(lats)

    # Variable 'aoi' als Dictionary im gewünschten Format speichern
    aoi = {
        "west": west,
        "south": south,
        "east": east,
        "north": north
    }

    # Als GeoJSON Datei speichern
    with open('aoi.geojson', 'w') as f:
        json.dump(m.user_roi, f)

    print(f" - AOI als Dictionary im openEO Format gespeichert: {aoi}")
    print(f" - Export als 'aoi.geojson'.")

    # Visuelle Bestätigung mit fit_bounds
    map_confirmation = folium.Map()
    folium.Rectangle(
        bounds=[[south, west], [north, east]],
        color='red',
        fill=True,
        fill_opacity=0.1
    ).add_to(map_confirmation)

    map_confirmation.fit_bounds([[south, west], [north, east]])
    display(map_confirmation)
else:
    print("⚠️ Bitte zuerst eine Fläche auf der Karte zeichnen!")

# Hilfsfunktion, um das UTM-CRS (Koordinatenreferenzsystem) für einen gegebenen Längen- und Breitengrad zu erhalten
def get_utm_crs(longitude, latitude):
    utm_band = str(int(math.floor((longitude + 180) / 6) + 1))
    if len(utm_band) == 1:
        utm_band = '0' + utm_band
    if latitude >= 0:
        crs_code = 'EPSG:326' + utm_band # Nördliche Hemisphäre
    else:
        crs_code = 'EPSG:327' + utm_band # Südliche Hemisphäre
    return crs_code

if 'aoi' in locals():
    # Erstelle ein Shapely Polygon aus dem aoi-Dictionary
    # Die Reihenfolge der Koordinaten für ein Polygon ist (Längengrad, Breitengrad)
    # aoi = {'west': West, 'south': Süd, 'east': Ost, 'north': Nord}
    polygon_geom = Polygon([
        (aoi['west'], aoi['south']),
        (aoi['east'], aoi['south']),
        (aoi['east'], aoi['north']),
        (aoi['west'], aoi['north']),
        (aoi['west'], aoi['south']) # Schließe das Polygon
    ])

    # Erstelle eine GeoSeries aus dem Polygon
    gdf = gpd.GeoSeries([polygon_geom], crs='EPSG:4326') # Das ursprüngliche CRS ist WGS84

    # Ermittle den Schwerpunkt, um die UTM-Zone zu bestimmen
    centroid_lon = gdf.centroid.x.iloc[0]
    centroid_lat = gdf.centroid.y.iloc[0]

    # Ermittle das passende UTM-CRS
    utm_crs = get_utm_crs(centroid_lon, centroid_lat)

    # Reprojektion zum UTM-CRS
    gdf_proj = gdf.to_crs(utm_crs)

    # Berechne die Fläche in Quadratmetern
    area_sq_m = gdf_proj.area.iloc[0]
    area_sq_km = area_sq_m / 1_000_000 # Umrechnung in Quadratkilometer

    print(f"Berechnete Fläche der Bounding Box:")
    print(f"  - {area_sq_m:.2f} m²")
    print(f"  - {area_sq_km:.4f} km²")
else:
    print("⚠️ Die Variable 'aoi' wurde nicht gefunden. Bitte eine Bounding Box einzeichnen.")

 - AOI als Dictionary im openEO Format gespeichert: {'west': -75.766397, 'south': -14.08967, 'east': -75.759015, 'north': -14.085133}
 - Export als 'aoi.geojson'.


Berechnete Fläche der Bounding Box:
  - 399923.06 m²
  - 0.3999 km²
